In [1]:
# This library downloads web pages.
import requests
# This library reads HTML code.
from bs4 import BeautifulSoup
# This library stores collected information in a table.
import pandas as pd
# This library pauses between requests.
import time
# This library records the collection date and time.
from datetime import datetime
# This library helps create folders and file paths.
import os

In [2]:
# Steam website
BASE_URL = "https://store.steampowered.com/search/"

# Three selected game categories
CATEGORIES = {
    "Action": {
        "tags": "19"
    },
    "Adventure": {
        "tags": "21"
    },
    "Singleplayer": {
        "tags": "4182"
    }
}

# Headers used when sending requests to Steam
# Replace the contact information with a valid course contact email
HEADERS = {
    "User-Agent": "B32919-Group7-UCU"
}

# Assignment requirement: top 500 ranked items
TARGET_ITEMS = 500

# Approximate number of games displayed per page
ITEMS_PER_PAGE = 50

# Folder for saving collected data
DATA_FOLDER = "data"


In [3]:
os.makedirs(DATA_FOLDER, exist_ok=True)

print("Project configuration completed.")
print("Selected categories:", ", ".join(CATEGORIES.keys()))
print("Target items:", TARGET_ITEMS)
print("Data folder:", DATA_FOLDER)

Project configuration completed.
Selected categories: Action, Adventure, Singleplayer
Target items: 500
Data folder: data


In [4]:
# Check Steam's robots.txt before scraping
ROBOTS_URL = "https://store.steampowered.com/robots.txt"

try:
    robots_response = requests.get(
        ROBOTS_URL,
        headers=HEADERS,
        timeout=20
    )

    robots_response.raise_for_status()

    print("Steam robots.txt retrieved successfully.")
    print("\nRules to review before scraping:\n")
    print(robots_response.text[:3000])

except requests.exceptions.RequestException as error:
    print("Could not retrieve Steam robots.txt.")
    print("Error:", error)

Steam robots.txt retrieved successfully.

Rules to review before scraping:

Host: store.steampowered.com
User-Agent: *
Disallow: /share/
Disallow: /news/externalpost/
Disallow: /account/emailoptout/?*token=
Disallow: /account/notificationsettings/?*token=
Disallow: /login/?*guestpasskey=
Disallow: /join/?*redir=
Disallow: /account/ackgift/
Disallow: /email/
Disallow: /widget/



In [5]:

# Create an empty list to store the collected games.
games = []

# Record the date and time when scraping begins.
scraped_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Track game IDs to avoid collecting duplicates.
seen_app_ids = set()

# Count the games collected in each category.
category_counts = {}

# Select the first three categories from CATEGORIES.
selected_categories = list(CATEGORIES.items())[:3]

# Check that three categories are available.
if len(selected_categories) != 3:
    raise ValueError("CATEGORIES must contain at least three categories.")

# Set the target number of games for each category.
category_targets = {
    selected_categories[0][0]: 168,
    selected_categories[1][0]: 166,
    selected_categories[2][0]: 166
}

# Display the collection targets.
print("GAME COLLECTION TARGETS")
print("-" * 35)

for category, target in category_targets.items():
    print(f"{category}: {target} games")

print(f"Total target: {sum(category_targets.values())} games")

# Scrape games from each of the three categories.
for category, settings in selected_categories:

    print(f"\nScraping category: {category}")

    # Initialize the category counter.
    category_counts[category] = 0

    # Start at the first page.
    start = 0

    # Continue until the category reaches its assigned target.
    while category_counts[category] < category_targets[category]:

        # Define the search parameters.
        params = {
            "filter": "topsellers",
            "tags": settings["tags"],
            "supportedlang": "english",
            "start": start
        }

        # Request the current page from Steam.
        try:
            response = requests.get(
                BASE_URL,
                params=params,
                headers=HEADERS,
                timeout=30
            )

            response.raise_for_status()

        except requests.exceptions.RequestException as error:
            print(
                f"Request failed for {category}, "
                f"start={start}: {error}"
            )

            # Wait before stopping this category.
            time.sleep(5)
            break

        # Parse the returned HTML.
        soup = BeautifulSoup(response.text, "html.parser")

        # Find the game rows on the page.
        rows = soup.select("a.search_result_row")

        # Display page progress.
        print(
            f"Category: {category} | "
            f"Page position: {start} | "
            f"Rows found: {len(rows)}"
        )

        # Stop if no games are found.
        if not rows:
            print(f"No more games found in {category}.")
            break

        # Process each game row.
        for row in rows:

            # Extract the Steam application ID.
            app_id = row.get("data-ds-appid", "").strip()

            # Skip missing or previously collected IDs.
            if not app_id or app_id in seen_app_ids:
                continue

            # Extract the game title.
            title_tag = row.select_one(".title")
            title = (
                title_tag.get_text(strip=True)
                if title_tag else ""
            )

            # Extract the release date.
            release_tag = row.select_one(".search_released")
            release_date = (
                release_tag.get_text(" ", strip=True)
                if release_tag else ""
            )

            # Extract the review summary.
            review_tag = row.select_one(".search_review_summary")
            review_summary = (
                review_tag.get("data-tooltip-html", "")
                if review_tag else ""
            )

            # Clean HTML line breaks from the review summary.
            review_summary = (
                review_summary
                .replace("<br>", " ")
                .replace("<br/>", " ")
                .replace("<br />", " ")
            )

            # Extract the visible price.
            price_tag = row.select_one(".search_price")
            price_text = (
                price_tag.get_text(" ", strip=True)
                if price_tag else ""
            )

            # Extract the discount percentage.
            discount_tag = row.select_one(".search_discount span")
            discount = (
                discount_tag.get_text(strip=True)
                if discount_tag else ""
            )

            # Extract the original price.
            original_tag = row.select_one(
                ".discount_original_price"
            )
            original_price = (
                original_tag.get_text(strip=True)
                if original_tag else ""
            )

            # Extract the final discounted price.
            final_tag = row.select_one(".discount_final_price")
            final_price = (
                final_tag.get_text(strip=True)
                if final_tag else ""
            )

            # Determine the current price.
            current_price = (
                final_price if final_price else price_text
            )

            # Extract supported platform names.
            platform_tags = row.select(".platform_img")

            platform_list = [
                class_name
                for tag in platform_tags
                for class_name in tag.get("class", [])
                if class_name != "platform_img"
            ]

            # Remove duplicate platform names.
            platform_list = list(dict.fromkeys(platform_list))

            # Combine platform names into one string.
            platforms = ", ".join(platform_list)

            # Record the application ID to prevent duplicates.
            seen_app_ids.add(app_id)

            # Increase the category count.
            category_counts[category] += 1

            # Store the collected game information.
            games.append({
                "app_id": app_id,
                "rank": len(games) + 1,
                "category": category,
                "title": title,
                "price": current_price,
                "original_price": original_price,
                "discount_percent": discount,
                "release_date": release_date,
                "review_summary": review_summary,
                "platforms": platforms,
                "scraped_at": scraped_at
            })

            # Display the collection progress.
            print(
                f"{category}: "
                f"{category_counts[category]}/"
                f"{category_targets[category]} games collected"
            )

            # Stop when this category reaches its target.
            if category_counts[category] >= category_targets[category]:
                break

        # Stop processing pages if the category target is reached.
        if category_counts[category] >= category_targets[category]:
            break

        # Stop if the page contains fewer results than expected.
        if len(rows) < ITEMS_PER_PAGE:
            print(f"Last or partial page reached for {category}.")
            break

        # Move to the next page.
        start += ITEMS_PER_PAGE

        # Wait before requesting the next page.
        time.sleep(3)

# Display the final collection summary.
print("\n" + "=" * 40)
print("SCRAPING COMPLETED")
print("=" * 40)

print(f"Total unique games collected: {len(games)}")

print("\nGames collected per category:")

for category, target in category_targets.items():
    collected = category_counts.get(category, 0)

    print(
        f"{category}: {collected}/{target} games"
    )

# Check whether the full target was reached.
if len(games) == 500:
    print("\nSUCCESS: All 500 unique games were collected.")
else:
    print(
        f"\nNOTICE: {500 - len(games)} games were not collected. "
        "Check the category counts and any request errors."
    )



GAME COLLECTION TARGETS
-----------------------------------
Action: 168 games
Adventure: 166 games
Singleplayer: 166 games
Total target: 500 games

Scraping category: Action
Category: Action | Page position: 0 | Rows found: 50
Action: 1/168 games collected
Action: 2/168 games collected
Action: 3/168 games collected
Action: 4/168 games collected
Action: 5/168 games collected
Action: 6/168 games collected
Action: 7/168 games collected
Action: 8/168 games collected
Action: 9/168 games collected
Action: 10/168 games collected
Action: 11/168 games collected
Action: 12/168 games collected
Action: 13/168 games collected
Action: 14/168 games collected
Action: 15/168 games collected
Action: 16/168 games collected
Action: 17/168 games collected
Action: 18/168 games collected
Action: 19/168 games collected
Action: 20/168 games collected
Action: 21/168 games collected
Action: 22/168 games collected
Action: 23/168 games collected
Action: 24/168 games collected
Action: 25/168 games collected
Action:

In [6]:

# Import the required libraries.
import os
import pandas as pd
from datetime import datetime

# Display the number of unique games collected from each category.
print("\nGames collected by category:")
print("-" * 40)

for category, target in category_targets.items():
    count = category_counts.get(category, 0)
    print(f"{category}: {count}/{target} games")

# Convert the collected game information into a DataFrame.
games_df = pd.DataFrame(games)

# Create the data folder if it does not already exist.
os.makedirs(DATA_FOLDER, exist_ok=True)

# Generate today's date for the output filename.
today = datetime.now().strftime("%Y-%m-%d")

# Create the CSV filename inside the data folder.
output_file = os.path.join(
    DATA_FOLDER,
    f"steam_top_sellers_{today}.csv"
)

# Save the dataset to the CSV file.
games_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

# Display the total number of unique games collected.
print("\nTotal unique games collected:", len(games_df))

# Display the dataset dimensions.
print("Dataset shape:", games_df.shape)

# Display the full path of the saved CSV file.
print("Saved file:", os.path.abspath(output_file))

# Verify whether exactly 500 games were collected.
if len(games_df) == 500:
    print("\nSUCCESS: Exactly 500 unique games were collected.")
else:
    print(
        f"\nWARNING: Expected 500 games, but collected "
        f"{len(games_df)} games."
    )

# Verify the number of games collected in each category.
print("\nCategory target verification:")
print("-" * 40)

for category, target in category_targets.items():
    count = category_counts.get(category, 0)

    if count == target:
        print(f"{category}: Target reached ({count}/{target})")
    else:
        print(f"{category}: Target not reached ({count}/{target})")

# Display the first five records.
display(games_df.head())




Games collected by category:
----------------------------------------
Action: 168/168 games
Adventure: 166/166 games
Singleplayer: 166/166 games

Total unique games collected: 500
Dataset shape: (500, 11)
Saved file: c:\Users\User\OneDrive\Desktop\THE TRIPOD PROJECT\Real Steam folder\Day 1\data\steam_top_sellers_2026-10-14.csv

SUCCESS: Exactly 500 unique games were collected.

Category target verification:
----------------------------------------
Action: Target reached (168/168)
Adventure: Target reached (166/166)
Singleplayer: Target reached (166/166)


,app_id,rank,category,title,price,original_price,discount_percent,release_date,review_summary,platforms,scraped_at
0,570,1,Action,Dota 2,Free,,,"9 Jul, 2013","Very Positive 87% of the 841,501 user reviews ...","win, mac, linux",2026-10-14 00:26:41
1,3393110,2,Action,AION 2,Free,,,"5 Oct, 2026","Mixed 54% of the 14,022 user reviews for this ...",win,2026-10-14 00:26:41
2,730,3,Action,Counter-Strike 2,Free,,,"21 Aug, 2012","Very Positive 85% of the 2,632,865 user review...","win, linux",2026-10-14 00:26:41
3,2828710,4,Action,Valor Mortis,$38.23,$44.98,,"13 Oct, 2026",Very Positive 83% of the 67 user reviews for t...,win,2026-10-14 00:26:41
4,2807960,5,Action,Battlefield™ 6,$27.99,$69.99,,"10 Oct, 2025","Mixed 65% of the 156,196 user reviews for this...",win,2026-10-14 00:26:41
